# Exact OpenAlex ids for the paper training corpus — re-extraction, proof, report

Interactive driver for the two scripts in this folder. Run the cells in order on a compute node
(an interactive job with ≥4 CPUs and ≥40 GB is enough; the login node is not).

| step | what | script | time (4 CPUs) |
|---|---|---|---|
| 1 | re-read the 405 tip shards with `id`, same filter as `Works_oa.py` | `extract_work_ids.py` | ~2–3 h, resumable per shard |
| 2 | per year: prove the row multiset equals `work_{year}.parquet`, attach ids by exact key, write `work_by_year_with_id_exact/`, write CSV + tex | `verify_attach_ids.py` | ~30–60 min |
| 3 | look at the result | this notebook | – |

Why this exists, and why the proof is on sets rather than positions, is written in the report the
run produces: `work_id_exact_verification.tex` (compile with `pdflatex`). In short: the training
files have no id because `Works_oa.py` did not read it; the title+year recovery in
`work_by_year_with_id/` covers ~90 % and sometimes links the wrong work; re-reading the same dump
with the same filter gives every training row its own id and changes nothing about what was
trained on, so no retraining follows.

**After this notebook:** `Atypicality/Data check/abstract_consistency_check.ipynb` (audit) and
`Atypicality/build_ppp_df_wabstract_rev.ipynb` (revised PPP file) both read
`OpenAlex/Data/work_by_year_with_id_exact/` and can be run in this session.

In [1]:
import os, sys, time, importlib
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

HERE = "/project/jevans/Dawoon/Science of Science/OpenAlex/Abstract data"
os.chdir(HERE); sys.path.insert(0, HERE)
import extract_work_ids as E
import verify_attach_ids as V

# workers: one shard per process, ~4 GB peak each. Use what the job gives, cap at 8.
WORKERS = min(8, int(os.environ.get("SLURM_CPUS_PER_TASK") or os.environ.get("SLURM_CPUS_ON_NODE") or os.cpu_count() or 4))
SMOKE_SHARDS = None      # e.g. 3 for a quick test of step 1 (step 2 then reports "NOT identical", as it should)
print(f"workers={WORKERS}   node={os.uname().nodename}")
print("mem cap (cgroup):", end=" ")
for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if os.path.exists(p):
        v = open(p).read().strip(); print(v if v == "max" else f"{int(v)/2**30:.0f} GiB"); break
else:
    print("unknown")

workers=4   node=midway3-0290.rcc.local
mem cap (cgroup): 8589934592 GiB


## 1. Re-extract with `id` (resumable: shards already written are skipped)

In [2]:
t0 = time.time()
argv = sys.argv; sys.argv = ["extract_work_ids.py", "--workers", str(WORKERS)] + (["--limit", str(SMOKE_SHARDS)] if SMOKE_SHARDS else [])
try:
    rc = E.main()
finally:
    sys.argv = argv
print(f"\nexit {rc} in {time.time()-t0:.0f}s")
log = pd.read_csv(os.path.join(os.path.dirname(E.OUT_DIR), "extract_log.csv"))
done = sorted(int(f[5:9]) for f in os.listdir(E.OUT_DIR) if f.endswith(".parquet"))
print(f"shards on disk: {len(done)} / 405")
display(log[log.status == "ok"][["n_read", "n_filtered", "n_kept", "sec"]].describe().T)

reconstruct_abstract loaded from Works_oa.py:
    def reconstruct_abstract(inv_index) -> str | None:
        """Safely reconstruct OpenAlex abstract from inverted index."""
        if inv_index is None or (isinstance(inv_index, float) and pd.isna(inv_index)):
    ...
405 shards, 4 workers -> /project/jevans/Dawoon/OpenAlex/Data/work_id_exact/shards
  10/405 shards  kept so far 3,323,522  (149s)
  20/405 shards  kept so far 6,793,744  (249s)
  30/405 shards  kept so far 9,919,481  (384s)
  40/405 shards  kept so far 13,128,800  (487s)
  50/405 shards  kept so far 16,317,152  (614s)
  60/405 shards  kept so far 19,462,920  (741s)
  70/405 shards  kept so far 22,526,353  (864s)
  80/405 shards  kept so far 25,653,410  (956s)
  90/405 shards  kept so far 28,636,884  (1078s)
  100/405 shards  kept so far 31,680,451  (1170s)
  110/405 shards  kept so far 34,318,131  (1272s)
  120/405 shards  kept so far 36,875,911  (1376s)
  130/405 shards  kept so far 39,677,626  (1480s)
  140/405 shards  k

,count,mean,std,min,25%,50%,75%,max
n_read,405.0,603387.269136,235350.316312,4.0,391305.0,591720.0,814696.0,1104287.0
n_filtered,405.0,390513.158025,94500.761792,3.0,335354.0,399839.0,447040.0,685909.0
n_kept,405.0,246052.716049,85404.060854,0.0,193487.0,261671.0,291454.0,663069.0
sec,405.0,35.204198,11.357569,0.1,27.9,37.9,43.0,78.8


## 2. Prove the row set is unchanged, attach ids, write the report

Per year: row counts, multiset equality of the `abstract` column and of the full key, exact-key
join (duplicate-record groups flagged `id_ambiguous`), comparison with the title+year ids. Writes
`OpenAlex/Data/work_by_year_with_id_exact/work_{year}.parquet`, `work_id_exact_verification.csv`,
`work_id_exact_disagreements.csv`, `work_id_exact_verification.tex`.

Set `os.environ["NB_YEARS"] = "1976-1977"` first to test on two years (report marked *partial*).

In [ ]:
os.environ.pop("NB_YEARS", None)          # all years; set e.g. "1976-1977" for a partial test
assert len([f for f in os.listdir(E.OUT_DIR) if f.endswith(".parquet")]) == 405 or os.environ.get("NB_YEARS"), \
    "not all 405 shards are present — finish step 1 first (or set NB_YEARS for a deliberately partial run)"
t0 = time.time()
importlib.reload(V)
rc = V.main()
print(f"\nexit {rc} in {time.time()-t0:.0f}s")

56 years
  1970: train   258,018  re-extract   258,018  abs-set =  key-set =  unmatched 0  ambiguous 877  title-agree 231,593 disagree 0 gain 26,425  (171.6s)
  1971: train   244,676  re-extract   244,676  abs-set =  key-set =  unmatched 0  ambiguous 506  title-agree 219,270 disagree 0 gain 25,406  (156.5s)
  1972: train   265,549  re-extract   265,549  abs-set =  key-set =  unmatched 0  ambiguous 589  title-agree 237,358 disagree 0 gain 28,191  (156.8s)
  1973: train   283,288  re-extract   283,288  abs-set =  key-set =  unmatched 0  ambiguous 638  title-agree 254,930 disagree 0 gain 28,358  (159.5s)
  1974: train   292,302  re-extract   292,302  abs-set =  key-set =  unmatched 0  ambiguous 563  title-agree 262,643 disagree 0 gain 29,659  (160.4s)
  1975: train   340,888  re-extract   340,888  abs-set =  key-set =  unmatched 0  ambiguous 1,163  title-agree 308,058 disagree 0 gain 32,830  (216.8s)
  1976: train   365,036  re-extract   365,036  abs-set =  key-set =  unmatched 0  ambiguo

## 3. Result

In [ ]:
v = pd.read_csv("work_id_exact_verification.csv")
tot = v[["n_train", "n_reextract", "n_unmatched", "n_ambiguous", "agree", "disagree", "gain_exact_only", "title_only"]].sum()
print(f"years: {len(v)} ({v.year.min()}-{v.year.max()})")
print(f"rows: train {int(tot.n_train):,}  re-extract {int(tot.n_reextract):,}")
print(f"abstract multiset equal in every year: {bool(v.abstract_multiset_equal.all())}")
print(f"full-key multiset equal in every year : {bool(v.key_multiset_equal.all())}")
print(f"unmatched rows: {int(tot.n_unmatched):,}   ambiguous (duplicate-record groups): {int(tot.n_ambiguous):,} "
      f"({tot.n_ambiguous/max(tot.n_train,1):.4%})")
print(f"vs title+year ids: agree {int(tot.agree):,}  disagree {int(tot.disagree):,}  exact-only gain {int(tot.gain_exact_only):,}  title-only {int(tot.title_only):,}")
verdict = v.abstract_multiset_equal.all() and v.key_multiset_equal.all() and tot.n_unmatched == 0
print("\nVERDICT:", "IDENTICAL row set — ids attached to the very rows the chains were trained on; no retraining implied."
      if verdict else "NOT identical in at least one year — see the table; investigate before concluding anything.")
display(v[["year", "n_train", "n_reextract", "abstract_multiset_equal", "key_multiset_equal", "n_unmatched", "n_ambiguous",
           "title_coverage", "agree", "disagree", "gain_exact_only", "sec"]])
dis = pd.read_csv("work_id_exact_disagreements.csv")
print(f"\n{len(dis)} sampled title-vs-exact disagreements (up to 5 per year):")
display(dis.head(15))
print("\nreport:", os.path.join(HERE, "work_id_exact_verification.tex"), " -> pdflatex work_id_exact_verification.tex")